In [ ]:
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.optim.lr_scheduler import ReduceLROnPlateau

from ugdatalab.utils.compose import Compose
from ugdatalab.models.galaxy_zoo import GalaxyZooGPUDataset
from ugdatalab.models.galaxy_zoo.constants import (
    N_LABELS,
    LABEL_COLUMNS,
    LABEL_TREE,
)
from ugdatalab.methods.neural_network.cnn import train_cnn
from ugdatalab.methods.neural_network.augmentation_gpu import GpuCenterCrop, GpuRandomRotation360

from architectures import build_resnet18, build_custom_cnn
import plotters

# Galaxy Image Classification — Class-Weighted Loss (Task 22, extra credit)

## Motivation

The 37 Galaxy Zoo 2 labels are *not* independent: they are organised in a decision tree (Willett et al. 2013 Figure 1) where each branch is only answered if its parent question was selected. For example, the "odd: lens/arc" vote fraction (Class8.2) is only meaningful for galaxies that were first identified as having an odd feature (Class6.1) — a galaxy classified with high confidence as *not* odd (Class6.2 ≈ 1) cannot also reasonably have high vote fractions for any Class8.x sub-feature. The vanilla RMSE loss we use elsewhere ignores this structure: it treats all 37 labels as equally important and penalises errors on rare deep-tree labels just as heavily as errors on the dominant top-level labels.

The lab manual's Task 22 (extra credit) proposes two ways to encode the tree dependencies:

(a) Add a sigmoid layer at the end and *re-weight* the outputs through a function that walks the decision tree, multiplying each child by its parent's prediction so that low-confidence parents suppress their children's outputs.

(b) Use a Weighted Binary Cross-Entropy or class-weighted KL divergence loss that *weights each label's contribution by the parent's probability* during training, so children of low-probability parents contribute less to the gradient.

We implement **option (b)** as it is a pure loss-function change — the model architecture is identical to the best model from `05b-augmentation.ipynb`, only `criterion` changes. This makes the comparison clean: any change in per-label performance is attributable to the loss, not to architectural differences. Concretely we use a **tree-weighted RMSE**:

$$L_{\mathrm{tree}} = \sqrt{\frac{\sum_i \sum_j w_{ij} (\hat y_{ij} - y_{ij})^2}{\sum_i \sum_j w_{ij}}}, \qquad w_{ij} = \max\left(\sum_{p\in\mathrm{parents}(j)} y_{ip},\; \epsilon\right),$$

where the sum over parents is over labels that gate label $j$ in the GZ2 decision tree. Root labels (Class1.x, Class6.1, Class6.2) have weight 1. A small floor $\epsilon = 10^{-3}$ keeps the gradient from vanishing for galaxies with $y_p = 0$, which would otherwise make those samples invisible to the optimiser.

## Build the parent map from `LABEL_TREE`

In [ ]:
name_to_idx = {name: i for i, name in enumerate(LABEL_COLUMNS)}

# For each label, list of parent label indices (empty for roots)
parent_indices = [[] for _ in range(N_LABELS)]
for parent_name, children in LABEL_TREE.items():
    p_idx = name_to_idx[parent_name]
    for child in children:
        parent_indices[name_to_idx[child]].append(p_idx)

# Inspect the resulting structure
parent_table = pd.DataFrame([
    {
        "label": LABEL_COLUMNS[i],
        "n_parents": len(parent_indices[i]),
        "parents": [LABEL_COLUMNS[p] for p in parent_indices[i]] or ["<root>"],
    }
    for i in range(N_LABELS)
])
parent_table

## Define the tree-weighted RMSE loss

The loss must remain a smooth, differentiable function of the model output. We build an `(N_LABELS, N_LABELS)` indicator matrix `M` such that `(M @ y.T)[j]` is the sum of parent vote fractions for label `j`. For root labels we set the corresponding row to give weight 1.

In [ ]:
EPS = 1e-3

# (N_LABELS, N_LABELS) selector matrix: row j has 1s at parent positions of label j
parent_selector = torch.zeros(N_LABELS, N_LABELS)
for j, parents in enumerate(parent_indices):
    for p in parents:
        parent_selector[j, p] = 1.0
# Roots: no parents -> we'll add a constant 1 in the loss directly.
is_root = torch.tensor([len(p) == 0 for p in parent_indices], dtype=torch.float32)


def tree_weighted_rmse(pred, target):
    """Tree-weighted root-mean-square error for GZ2 multi-label regression.

    Parameters
    ----------
    pred : torch.Tensor, shape (B, N_LABELS)
    target : torch.Tensor, shape (B, N_LABELS)
    """
    sel = parent_selector.to(target.device)
    root = is_root.to(target.device)
    # Per-sample, per-label weight = sum of parent true values + root indicator + EPS
    weights = target @ sel.T + root.unsqueeze(0) + EPS
    sq = (pred - target) ** 2
    return torch.sqrt((weights * sq).sum() / weights.sum())

## Load data and reconstruct the best model from NB 04/05

In [ ]:
img_data = np.load("artifacts/galaxy_zoo_images.npz")
images = img_data["images"]
label_data = np.load("artifacts/galaxy_zoo_labels.npz")
labels = label_data["labels"]
split_data = np.load("artifacts/split_indices.npz")
train_idx, val_idx = split_data["train_idx"], split_data["val_idx"]

train_images, val_images = images[train_idx], images[val_idx]
train_labels, val_labels = labels[train_idx], labels[val_idx]
INPUT_SIZE = 96
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Class-weighted loss experiment targets the Custom CNN by request.
custom_data = np.load("artifacts/custom_result.npz", allow_pickle=True)
BEST_MODEL_NAME = "Custom CNN"
_n_channels = [int(x) for x in custom_data["n_channels_list"]]
_kernels = [int(x) for x in custom_data["kernel_sizes"]]
_fc_sizes = [int(x) for x in custom_data["fc_sizes"]]
# dropout_rate may be scalar (uniform) or per-FC-layer list — handle both
_dropout = [float(x) for x in np.atleast_1d(custom_data["dropout_rate"])]
_pool = str(custom_data["pool_type"])

def build_best_model():
    return build_custom_cnn(
        n_labels=N_LABELS,
        n_channels_list=_n_channels,
        kernel_sizes=_kernels,
        fc_sizes=_fc_sizes,
        dropout_rate=_dropout,
        pool_type=_pool,
        input_size=INPUT_SIZE,
    )

print(f"Target model: {BEST_MODEL_NAME}")


## Train with the tree-weighted loss

We use the same recipe as the best augmented run in `05b-augmentation.ipynb` (rotation augmentation + `ReduceLROnPlateau` scheduler) so that the only change relative to the baseline is the loss function.

In [ ]:
from tqdm.auto import tqdm

from ugdatalab.methods.neural_network.cnn import count_parameters

BATCH_SIZE = 1024
default_transform = Compose([GpuCenterCrop(INPUT_SIZE)])
aug_transform = Compose([GpuRandomRotation360(), GpuCenterCrop(INPUT_SIZE)])
train_batches = GalaxyZooGPUDataset(
    train_images, train_labels, batch_size=BATCH_SIZE, transform=aug_transform,
    device=DEVICE, shuffle=True,
)
val_batches = GalaxyZooGPUDataset(
    val_images, val_labels, batch_size=BATCH_SIZE, transform=default_transform,
    device=DEVICE, shuffle=False,
)


def _run_epoch_custom(model, batches, loss_fn, optimizer, device, training):
    """One epoch of custom-loss training or evaluation."""
    model.train() if training else model.eval()
    total = 0.0
    n_batches = 0
    ctx = torch.enable_grad() if training else torch.no_grad()
    is_cuda = device == "cuda"
    with ctx:
        for images, targets in batches:
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", dtype=torch.bfloat16, enabled=is_cuda):
                pred = model(images)
                loss = loss_fn(pred, targets)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total += float(loss.item())
            n_batches += 1
    return total / max(n_batches, 1)


def train_with_loss(model, loss_fn, n_epochs, lr, seed):
    """Mirror of train_cnn but with a pluggable loss function."""
    device = DEVICE
    torch.manual_seed(seed)
    torch.backends.cudnn.benchmark = True
    model = model.to(device)
    # See cnn.py:train_cnn for the rationale on keeping `model` as the
    # state_dict source and only using `compiled_model` for execution.
    compiled_model = torch.compile(model) if device == "cuda" else model
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, fused=True)
    scheduler = ReduceLROnPlateau(optimizer, factor=0.5, patience=3)

    train_losses = np.empty(n_epochs)
    val_losses = np.empty(n_epochs)
    learning_rates = np.empty(n_epochs)
    best_state, best_val, best_ep = None, float("inf"), 0
    for epoch in tqdm(range(n_epochs), desc="Training (tree-weighted)"):
        learning_rates[epoch] = optimizer.param_groups[0]["lr"]
        train_losses[epoch] = _run_epoch_custom(compiled_model, train_batches, loss_fn, optimizer, device, training=True)
        val_losses[epoch] = _run_epoch_custom(compiled_model, val_batches, loss_fn, None, device, training=False)
        scheduler.step(val_losses[epoch])
        if val_losses[epoch] < best_val:
            best_val = val_losses[epoch]
            best_ep = epoch
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    return SimpleNamespace(
        train_losses=train_losses,
        val_losses=val_losses,
        best_epoch=best_ep,
        best_val_loss=best_val,
        model_state=best_state,
        n_parameters=count_parameters(model),
        learning_rates=learning_rates,
    )


_ckpt_pt = Path("artifacts/class_weighted.pt")
_ckpt_npz = Path("artifacts/class_weighted_result.npz")
if _ckpt_pt.exists() and _ckpt_npz.exists():
    _data = np.load(_ckpt_npz)
    cw_result = SimpleNamespace(
        model_state=torch.load(_ckpt_pt, map_location=DEVICE),
        train_losses=_data["train_losses"],
        val_losses=_data["val_losses"],
        best_epoch=int(_data["best_epoch"]),
        best_val_loss=float(_data["best_val_loss"]),
        n_parameters=int(_data["n_parameters"]),
        learning_rates=_data["learning_rates"],
    )
    print(f"Loaded cached class_weighted.pt (best tree-weighted val RMSE: {cw_result.best_val_loss:.4f})")
else:
    cw_result = train_with_loss(
        build_best_model(), tree_weighted_rmse,
        n_epochs=100, lr=1e-3, seed=42,
    )
    print(f"Best epoch: {cw_result.best_epoch + 1}")
    print(f"Best tree-weighted validation RMSE: {cw_result.best_val_loss:.4f}")
    torch.save(cw_result.model_state, _ckpt_pt)
    np.savez_compressed(
        _ckpt_npz,
        train_losses=cw_result.train_losses,
        val_losses=cw_result.val_losses,
        best_epoch=cw_result.best_epoch,
        best_val_loss=cw_result.best_val_loss,
        n_parameters=cw_result.n_parameters,
        learning_rates=cw_result.learning_rates,
    )
    print("Saved class_weighted.pt and class_weighted_result.npz")


## Loss curves

Note: the absolute value of the tree-weighted RMSE is **not directly comparable** to the unweighted RMSE elsewhere — the denominator differs. What matters is the *shape* of the curve and whether the model converges stably.

In [ ]:
axes = plotters.plot_loss_with_lr(
    cw_result.train_losses, cw_result.val_losses,
    cw_result.learning_rates,
    f"{BEST_MODEL_NAME} + tree-weighted RMSE",
)
plt.show()

## Did weighting change per-label performance?

To compare fairly with the standard-loss run we must evaluate both models with the *same* metric — the unweighted per-label RMSE on the validation set. Below we load the augmented-loss model from NB 05 and the class-weighted model, predict on the validation set, and compare per-label residuals.

In [ ]:
from ugdatalab.methods.neural_network.cnn import predict_cnn
from ugdatalab.models.galaxy_zoo.constants import LABEL_DESCRIPTIVE

# val_batches reused from the training cell above

# Standard-loss best augmented model
std_model = build_best_model()
std_model.load_state_dict(torch.load("artifacts/best_augmented.pt", map_location=DEVICE))
std_pred = predict_cnn(std_model, val_batches)

# Tree-weighted-loss model
cw_model = build_best_model()
cw_model.load_state_dict(cw_result.model_state)
cw_pred = predict_cnn(cw_model, val_batches)

std_rmse = np.sqrt(((std_pred - val_labels) ** 2).mean(axis=0))
cw_rmse = np.sqrt(((cw_pred - val_labels) ** 2).mean(axis=0))

comparison = pd.DataFrame({
    "label": [LABEL_DESCRIPTIVE[c] for c in LABEL_COLUMNS],
    "standard_rmse": std_rmse,
    "weighted_rmse": cw_rmse,
    "delta": cw_rmse - std_rmse,
}).sort_values("delta").reset_index(drop=True)

print(f"Standard-loss bulk val RMSE: {std_rmse.mean():.4f}")
print(f"Tree-weighted-loss bulk val RMSE: {cw_rmse.mean():.4f}")
print("\nLabels most improved by tree weighting (negative delta):")
display(comparison.head(8))
print("\nLabels most hurt by tree weighting (positive delta):")
display(comparison.tail(8).iloc[::-1])

## Discussion

Tree weighting redirects the optimiser's attention: errors on labels whose parent question was confidently answered "yes" carry more weight, while errors on labels whose parent was rarely selected (e.g. odd-feature sub-classes for non-odd galaxies) carry almost none. We therefore expect two effects:

1. **Top-level / well-sampled labels** (smooth, features/disk, edge-on, spiral) get more effective gradient signal and should improve, because every training galaxy contributes to their loss with weight close to 1.
2. **Deep-tree, rare labels** (lens/arc, dust lane, high arm counts) get less gradient signal in absolute terms because most galaxies have parent values close to 0 and therefore weights close to $\epsilon$. Their per-label RMSE may *increase* under the weighted loss — not because the model got worse, but because it stopped trying to fit the long tail of zeros and is now optimised for the regime where the labels are physically meaningful.

The bulk RMSE measured under the *unweighted* metric is therefore not the right success criterion for this experiment. The right read-out is whether the *shape* of the per-label RMSE distribution becomes more astrophysically defensible: better fits where the labels actually carry information, slightly worse fits where they were nearly constant zero. The comparison table above lets the reader make that judgement explicitly.